# TrafficWatch: Colab training runbook

Generated from the Colab runbook. Trains and compares the YOLOv8 detector for
the incident-detection pipeline. Run cells top to bottom on a **blank Colab runtime** with
a GPU (Runtime > Change runtime type > T4 or better). Nothing here needs a local install;
everything the repo's own tests check locally is re-verified here from scratch.

**Scope (decision D-11):** only `yolov8n` and `yolov8s` are trained and compared here. No
other detector architecture is in scope; that would be a logged scope-change decision the
author has not made.

**What this notebook does not do:** it does not evaluate on the official test split (that
is a separate, logged, one-time event in Phase 5), and it does not compute the T0 zero-shot
baseline (T0 runs the repo's own detector through the Phase 5 evaluation harness, not
Ultralytics training, and does not need a GPU).

**Before you start:**
1. `Runtime > Change runtime type` and select a GPU.
2. 3. Have the Phase 1 dataset ready: `data/processed/yolo_ds_v1.tar` and `.sha256` copied to
   `MyDrive/tid/data/` (produced locally by the repo's Phase 1 scripts, see the data-not-found
   guidance below if this has not been done yet).

## 1. Hardware and versions

In [ ]:
import platform
import subprocess
import sys

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print(sys.version)
print(platform.platform())

gpu_present = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
assert gpu_present, (
    "No GPU detected. Go to Runtime > Change runtime type, select a GPU, then rerun "
    "this cell before continuing."
)
print("GPU OK:", gpu_present)

## 2. Fetch the code

Clones the public repository anonymously (`--depth 1`). No token or secret is needed, because the
repository is public.

In [ ]:
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/dilconkihoro/traffic-incident-detection-public.git"
REPO_DIR = Path("/content/traffic-incident-detection")

shutil.rmtree(REPO_DIR, ignore_errors=True)
result = subprocess.run(
    ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
    capture_output=True,
    text=True,
)
assert result.returncode == 0, "git clone failed: " + result.stderr
print("cloned", REPO_URL)

## 3. Install requirements

Installs the pinned extras from `requirements-colab.txt`. Torch and torchvision are **not**
reinstalled: Colab's preinstalled build already matches its GPU and CUDA driver, and
installing our CPU-pinned `requirements.txt` here would silently replace it with a build
that cannot see the GPU.

In [ ]:
%cd /content/traffic-incident-detection
%pip install -q -r requirements-colab.txt

import torch
import ultralytics

print("ultralytics", ultralytics.__version__)
print("torch", torch.__version__, "cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "torch cannot see a GPU; check the runtime type"

## 4. Drive, paths, and data staging

Mounts Drive and looks for the dataset tar built by the repo's local Phase 1 scripts
(`scripts/acquire_datasets.py`, `scripts/eda_dataset.py`, `scripts/convert_to_yolo.py`,
packaged per `docs/03_DATA_PIPELINE.md` section 6). If it is missing, this cell prints the
exact local steps and stops rather than guessing at a path.

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
DRIVE = "/content/drive/MyDrive/tid"
DATA_VERSION = "yolo_ds_v1"
TAR = f"{DRIVE}/data/{DATA_VERSION}.tar"
SHA = f"{DRIVE}/data/{DATA_VERSION}.sha256"

In [ ]:
import pathlib

if not (pathlib.Path(TAR).is_file() and pathlib.Path(SHA).is_file()):
    raise SystemExit(
        f"Dataset tar not found at {TAR}.\n\n"
        "This notebook trains on a dataset built locally first, not on Colab. On your "
        "own machine, in the repo:\n"
        "  1. Place UA-DETRAC and DhakaAI archives in data/raw/ (docs/03 section 9 has the "
        "download steps).\n"
        "  2. Run scripts/acquire_datasets.py verify on each archive.\n"
        "  3. Run the EDA, conversion, and split scripts from docs/03 sections 9 (once "
        "Phase 1 is implemented) to build data/processed/yolo_ds_v1/.\n"
        f"  4. Package it and copy {DATA_VERSION}.tar and {DATA_VERSION}.sha256 to "
        f"MyDrive/tid/data/ on Drive.\n\n"
        "Then rerun this cell."
    )
print("found:", TAR)

In [ ]:
!mkdir -p /content/data
!cd {DRIVE}/data && sha256sum -c {DATA_VERSION}.sha256
!tar -xf {TAR} -C /content/data
!sed -i 's#^path:.*#path: /content/data/{DATA_VERSION}#' /content/data/{DATA_VERSION}/data.yaml
!cat /content/data/{DATA_VERSION}/data.yaml

## 5. Data characteristics

Cleaning, ignored-region masking, class mapping, and stride are already applied by the
local conversion step; this section only narrates what shipped in the tar, for report section 5.3.
It never recomputes or invents a number: every value below is read from `stats.json` and
`eda_insights.json`, and a missing field prints `VERIFY` instead of being guessed.

In [ ]:
import json
import pathlib

DATA_DIR = pathlib.Path(f"/content/data/{DATA_VERSION}")
stats = json.loads((DATA_DIR / "stats.json").read_text())
eda_path = DATA_DIR / "eda_insights.json"
eda = json.loads(eda_path.read_text()) if eda_path.is_file() else {}

print(json.dumps(stats, indent=2)[:2000])

In [ ]:
import matplotlib.pyplot as plt

class_counts = stats.get("class_distribution", {}).get("train", {})
if class_counts:
    plt.figure(figsize=(6, 3))
    plt.bar(list(class_counts.keys()), list(class_counts.values()))
    plt.title("Train boxes per class (Figure ID: assign in the figure registry)")
    plt.ylabel("boxes")
    plt.show()
else:
    print("VERIFY: stats.json has no class_distribution.train; check the converter output")

## 6. Smoke test (always run this before committing to a full run)

One epoch on about 200 images proves the data path and `data.yaml` are usable before
spending GPU time on 30 to 50 epochs (runbook section 2; decision D-21: no training of
any size runs on the local machine, so this smoke run also happens here, not locally).

In [ ]:
import random
import shutil

import yaml

SMOKE_DIR = pathlib.Path("/content/data/smoke")
shutil.rmtree(SMOKE_DIR, ignore_errors=True)
(SMOKE_DIR / "images").mkdir(parents=True)
(SMOKE_DIR / "labels").mkdir(parents=True)

train_images = sorted((DATA_DIR / "images" / "train").glob("*"))
assert train_images, f"no images found under {DATA_DIR / 'images' / 'train'}"
sample = random.Random(0).sample(train_images, min(200, len(train_images)))
for image in sample:
    label = DATA_DIR / "labels" / "train" / (image.stem + ".txt")
    shutil.copy(image, SMOKE_DIR / "images" / image.name)
    if label.is_file():
        shutil.copy(label, SMOKE_DIR / "labels" / label.name)

full_data = yaml.safe_load((DATA_DIR / "data.yaml").read_text())
smoke_yaml = SMOKE_DIR / "data.yaml"
smoke_yaml.write_text(
    yaml.safe_dump(
        {"path": str(SMOKE_DIR), "train": "images", "val": "images", "names": full_data["names"]}
    )
)
print(f"smoke set: {len(sample)} images at {SMOKE_DIR}")

In [ ]:
from ultralytics import YOLO

smoke = YOLO("yolov8n.pt")
smoke.train(data=str(smoke_yaml), epochs=1, imgsz=640, batch=8, seed=0, workers=2, plots=False)
print("smoke test passed: the data path and data.yaml are usable")

## 7. Check for an already-trained model

If a run already completed (`best.pt` and `model_card.json` both present under
`MyDrive/tid/runs/<RUN_ID>/`), list it here. There is no need to retrain a run that already
finished; jump to section 10 to compare and export what already exists.

In [ ]:
runs_dir = pathlib.Path(f"{DRIVE}/runs")
existing = []
if runs_dir.is_dir():
    for run_dir in sorted(runs_dir.iterdir()):
        card_path = run_dir / "model_card.json"
        weights_path = run_dir / "weights" / "best.pt"
        if weights_path.is_file() and card_path.is_file():
            card = json.loads(card_path.read_text())
            existing.append(card)
            print(run_dir.name, "-", card.get("val", "no val metrics recorded"))
if not existing:
    print("no completed runs found on Drive yet")

## 8. Run matrix

| Run | Model | Training data | Epochs | Purpose |
|---|---|---|---|---|
| T1 | yolov8n | UA-DETRAC (stride 5) | 30 | pipeline and baseline check, cheap |
| T2 | yolov8s | UA-DETRAC (stride 5) | 50 | in-domain main run |
| T3 | yolov8s | UA-DETRAC plus DhakaAI | 50 | tests the diversity claim; evaluated on both `val` and `dhaka_val` |
| T4 (optional) | best of T2 or T3 | same | 40 | small and distant vehicles only, off by default |

Hyperparameters not listed stay at Ultralytics defaults (runbook section 4). The test
split is never touched in this notebook. Each run: train, resume if disconnected, validate
on `val` only, write a model card, export to Drive.

In [ ]:
import datetime
import shutil

import torch
import ultralytics
from ultralytics import YOLO

SEED = 0
completed_runs = {}  # run_id -> {"card": dict, "val": ultralytics metrics, "model": YOLO}


def train_run(run_id, base_model, data_yaml, epochs, imgsz=640, batch=16):
    """Train one run, checkpointed to Drive every 5 epochs, resumable if interrupted."""
    run_dir = pathlib.Path(f"{DRIVE}/runs/{run_id}")
    last = run_dir / "weights" / "last.pt"
    if last.is_file():
        print(f"resuming {run_id} from {last}")
        model = YOLO(str(last))
        model.train(resume=True)
    else:
        model = YOLO(base_model)
        model.train(
            data=data_yaml,
            epochs=epochs,
            imgsz=imgsz,
            batch=batch,
            seed=SEED,
            deterministic=True,
            patience=15,
            save_period=5,
            workers=2,
            amp=True,
            plots=True,
            project=f"{DRIVE}/runs",
            name=run_id,
            exist_ok=True,
        )
    return model


def repo_git_sha():
    repo = pathlib.Path("/content/traffic-incident-detection")
    if (repo / ".git").is_dir():
        done = subprocess.run(
            ["git", "-C", str(repo), "rev-parse", "HEAD"],
            capture_output=True,
            text=True,
            check=True,
        )
        return done.stdout.strip()
    sha_file = repo / "GIT_SHA"
    assert sha_file.is_file(), (
        "No .git folder in the fetched code. Rerun section 2 to clone the repository."
    )
    return sha_file.read_text(encoding="utf-8").strip()


def validate_and_export(run_id, model, base_model, epochs_planned, data_yaml, extra_val=None):
    """Validate on val (and optionally another split), write model_card.json, zip to Drive."""
    m_val = model.val(data=data_yaml, split="val", imgsz=640, conf=0.001, iou=0.6, plots=True)
    run_dir = pathlib.Path(f"{DRIVE}/runs/{run_id}")
    card = {
        "run_id": run_id,
        "base_model": base_model,
        "class_names": list(model.names.values()),
        "imgsz": 640,
        "seed": SEED,
        "git_sha": repo_git_sha(),
        "epochs_planned": epochs_planned,
        "data_tar_sha256": pathlib.Path(SHA).read_text().split()[0],
        "ultralytics": ultralytics.__version__,
        "torch": torch.__version__,
        "gpu": torch.cuda.get_device_name(0),
        "val": {
            "map50": float(m_val.box.map50),
            "map50_95": float(m_val.box.map),
            "precision": float(m_val.box.mp),
            "recall": float(m_val.box.mr),
        },
        "date": datetime.date.today().isoformat(),
    }
    if extra_val:
        m_extra = model.val(
            data=data_yaml, split=extra_val, imgsz=640, conf=0.001, iou=0.6, plots=True
        )
        card[extra_val] = {
            "map50": float(m_extra.box.map50),
            "map50_95": float(m_extra.box.map),
            "precision": float(m_extra.box.mp),
            "recall": float(m_extra.box.mr),
        }
    (run_dir / "model_card.json").write_text(json.dumps(card, indent=2))
    pathlib.Path(f"{DRIVE}/export").mkdir(exist_ok=True)
    shutil.make_archive(f"{DRIVE}/export/{run_id}", "zip", run_dir)
    lock_path = run_dir / "requirements.lock"
    lock_path.write_text(subprocess.run(["pip", "freeze"], capture_output=True, text=True).stdout)
    completed_runs[run_id] = {"card": card, "model": model}
    print(f"{run_id}: val mAP50={card['val']['map50']:.3f} mAP50:95={card['val']['map50_95']:.3f}")
    return card

### T1: yolov8n on UA-DETRAC (30 epochs)

In [ ]:
import yaml

T1_ID = "T1_y8n_detrac_s5_e30_seed0"
data_yaml = str(DATA_DIR / "data.yaml")

detrac_train = sorted((DATA_DIR / "images" / "train").glob("*"))
assert len(detrac_train) == 13855, (
    f"expected 13855 UA-DETRAC train images, found {len(detrac_train)}; the default data.yaml "
    "also lists DhakaAI, so T1 and T2 must use the UA-DETRAC-only file written below"
)
detrac_data = yaml.safe_load(pathlib.Path(data_yaml).read_text())
detrac_data["train"] = ["images/train"]
detrac_yaml_path = pathlib.Path("/content/data/detrac_data.yaml")
detrac_yaml_path.write_text(yaml.safe_dump(detrac_data))
detrac_yaml = str(detrac_yaml_path)
print("UA-DETRAC-only training file:", detrac_yaml, "| git sha:", repo_git_sha())

if T1_ID in {c["run_id"] for c in existing}:
    print(f"{T1_ID} already completed on Drive; skipping training")
else:
    model_t1 = train_run(T1_ID, "yolov8n.pt", detrac_yaml, epochs=30)
    validate_and_export(T1_ID, model_t1, "yolov8n.pt", 30, detrac_yaml)

### T2: yolov8s on UA-DETRAC (50 epochs, main in-domain run)

In [ ]:
T2_ID = "T2_y8s_detrac_s5_e50_seed0"

if T2_ID in {c["run_id"] for c in existing}:
    print(f"{T2_ID} already completed on Drive; skipping training")
else:
    model_t2 = train_run(T2_ID, "yolov8s.pt", detrac_yaml, epochs=50)
    validate_and_export(T2_ID, model_t2, "yolov8s.pt", 50, detrac_yaml)

### T3: yolov8s on UA-DETRAC plus DhakaAI (50 epochs)

Needs `images/dhaka_train` and `labels/dhaka_train` in the tar (`docs/03` section 6). If
Phase 1 has not built the DhakaAI split yet, this cell explains that and stops instead of
training on the wrong data.

In [ ]:
import yaml

T3_ID = "T3_y8s_detrac_dhaka_s5_e50_seed0"
dhaka_train_dir = DATA_DIR / "images" / "dhaka_train"

if not dhaka_train_dir.is_dir():
    print(
        f"SKIPPING T3: {dhaka_train_dir} not found in this dataset version. "
        "T3 needs the DhakaAI training split; build it locally first (docs/03 section 6) "
        "and repackage the tar, then rerun from section 4."
    )
elif T3_ID in {c["run_id"] for c in existing}:
    print(f"{T3_ID} already completed on Drive; skipping training")
else:
    t3_yaml_path = pathlib.Path("/content/data/t3_data.yaml")
    t3_data = yaml.safe_load(pathlib.Path(data_yaml).read_text())
    t3_data["train"] = ["images/train", "images/dhaka_train"]
    t3_yaml_path.write_text(yaml.safe_dump(t3_data))

    model_t3 = train_run(T3_ID, "yolov8s.pt", str(t3_yaml_path), epochs=50)
    validate_and_export(T3_ID, model_t3, "yolov8s.pt", 50, str(t3_yaml_path), extra_val="dhaka_val")

### T4 (optional): higher resolution, only if T2 or T3 show a small-object weakness

Off by default. Uncomment and set `T4_BASE_RUN_ID` to the better of T2 or T3 only after
reviewing their per-class and box-size results; log the reason in
the training run log (runbook section 4).

In [ ]:
# T4_BASE_RUN_ID = T2_ID  # or T3_ID
# T4_ID = "T4_" + T4_BASE_RUN_ID.split('_', 1)[1].replace('e50', 'e40') + "_imgsz960"
# base_weights = f"{DRIVE}/runs/{T4_BASE_RUN_ID}/weights/best.pt"
# model_t4 = train_run(T4_ID, base_weights, data_yaml, epochs=40, imgsz=960, batch=8)
# validate_and_export(T4_ID, model_t4, base_weights, 40, data_yaml)

## 9. Compare runs and select

Selection rule (runbook section 4, evaluation protocol section 3): validation mAP50:95 first, then
recall at the operating confidence. Latency is not decided here; it is measured on the
author's local machine in Phase 4 and is the final tie-break before freezing a model, not a
Colab-side decision.

In [ ]:
import pandas as pd

rows = []
for run_id, info in completed_runs.items():
    card = info["card"]
    row = {"run_id": run_id, **card["val"]}
    if "dhaka_val" in card:
        row.update({f"dhaka_{k}": v for k, v in card["dhaka_val"].items()})
    rows.append(row)

if not rows:
    print("no completed runs this session; nothing to compare yet")
else:
    table = pd.DataFrame(rows).sort_values(["map50_95", "recall"], ascending=False)
    display(table)
    table.set_index("run_id")["map50_95"].plot.bar(
        title="Validation mAP50:95 by run (Figure ID: assign in the figure registry)",
        ylabel="mAP50:95",
    )
    best = table.iloc[0]["run_id"]
    print(
        f"\nRecommended on val mAP50:95 then recall: {best}\n"
        "Confirm with the local latency benchmark (Phase 4, docs/05 section 6) before freezing."
    )

## 10. What to do next (paste-back checklist)

The workspace does not log a training run, or use its numbers in any table, until these are pasted
back into the project record (project rules section 7):

1. For each completed run, download `MyDrive/tid/export/<RUN_ID>.zip` and unzip it to
   `weights/<RUN_ID>/` in the repo (needs `best.pt` and `model_card.json`; `requirements.lock`
   in the zip is the exact pinned environment that produced it, and belongs at the repo
   root once a run is selected as the reported one).
2. Paste `model_card.json` and `results.csv` for each run into the chat, or attach the files.
3. Capture the evidence screenshots listed in the runbook section 7 into `docs/evidence/`,
   named with the figure ID reserved in `docs/06`.
4. Report back which run you are selecting as the reported model, so it can be checked
   against the run log template in the training run log.

In [ ]:
print("Completed this session:", list(completed_runs))
print("Drive export directory:", f"{DRIVE}/export")